## tl;dr

- 样本覆盖 13 个代表产品、10 个能力维度；评分用于比较官网可证实的能力覆盖，不代表产品质量或市场份额。
- `2` = 官网证实为核心/完整能力，`1` = 部分覆盖或通用能力，`0` = 本轮公开资料未证实。
- 结论应与配套研究报告一起阅读；矩阵是可复算的证据层，不替代产品实测。


## Context & Methods

目标：判断 ContentFerry 不应在哪些成熟能力上重复竞争，以及哪些能力组合仍有差异化空间。

### Key Assumptions

- 截止日期：2026-07-16。
- 样本为目的性抽样，覆盖国内矩阵工具、公众号工具、海外社媒套件、AI 营销平台和通用 Agent/自动化平台，不用于估算市场份额。
- 仅依据官网、官方帮助中心、官方仓库与官方发布信息；没有公开证据不等于产品一定不具备该能力。
- “本地/自托管”按产品运行与数据控制能力评分，不把普通桌面客户端自动视为完整本地优先。


## Data

读取人工核验后的能力矩阵，并保留产品类别与来源链接。


In [1]:
from pathlib import Path
import pandas as pd

csv_path = Path.cwd() / 'competitor-capability-matrix.csv'
if not csv_path.exists():
    csv_path = Path.cwd() / 'research' / 'competitor-capability-matrix.csv'

df = pd.read_csv(csv_path)
capability_columns = [
    'research_hotspots', 'ai_creation', 'brand_context', 'platform_adaptation',
    'multi_account_publishing', 'approval_governance', 'quality_compliance',
    'analytics_inbox', 'agent_workflows', 'local_self_host'
]

assert len(df) == 13
assert df['product'].is_unique
assert df[capability_columns].isin([0, 1, 2]).all().all()
print(df[['product', 'region', 'archetype', *capability_columns]].to_string(index=False))


      product region archetype  research_hotspots  ai_creation  brand_context  platform_adaptation  multi_account_publishing  approval_governance  quality_compliance  analytics_inbox  agent_workflows  local_self_host
          蚁小二     中国    矩阵运营平台                  0            1              0                    1                         2                    1                   1                2                1                1
        新榜小豆芽     中国    矩阵运营平台                  1            1              0                    1                         2                    1                   0                2                0                1
       135编辑器     中国   公众号创作工具                  0            2              0                    1                         1                    1                   0                0                0                0
         壹伴助手     中国   公众号运营插件                  1            1              0                    1                         1        

## Results

先看各能力在样本中的覆盖广度，再看不同产品类别的结构性强弱。


In [2]:
capability_labels = {
    'research_hotspots': '研究/热点',
    'ai_creation': 'AI创作',
    'brand_context': '品牌/人设上下文',
    'platform_adaptation': '平台适配',
    'multi_account_publishing': '多账号发布',
    'approval_governance': '审批/治理',
    'quality_compliance': '质量/合规',
    'analytics_inbox': '分析/收件箱',
    'agent_workflows': 'Agent/工作流',
    'local_self_host': '本地/自托管',
}

coverage = pd.DataFrame({
    'capability': [capability_labels[c] for c in capability_columns],
    'core_count': [(df[c] == 2).sum() for c in capability_columns],
    'partial_count': [(df[c] == 1).sum() for c in capability_columns],
    'not_evidenced_count': [(df[c] == 0).sum() for c in capability_columns],
})
coverage['covered_count'] = coverage['core_count'] + coverage['partial_count']
coverage['coverage_rate'] = coverage['covered_count'] / len(df)
coverage = coverage.sort_values(['coverage_rate', 'core_count'], ascending=False).reset_index(drop=True)
print(coverage.to_string(index=False))


capability  core_count  partial_count  not_evidenced_count  covered_count  coverage_rate
      AI创作           3             10                    0             13       1.000000
      平台适配           5              6                    2             11       0.846154
    分析/收件箱           7              3                    3             10       0.769231
     审批/治理           4              6                    3             10       0.769231
     多账号发布           6              3                    4              9       0.692308
 Agent/工作流           4              3                    6              7       0.538462
     研究/热点           2              4                    7              6       0.461538
  品牌/人设上下文           2              3                    8              5       0.384615
     质量/合规           1              4                    8              5       0.384615
    本地/自托管           2              2                    9              4       0.307692


In [3]:
archetype_profile = (
    df.groupby('archetype')[capability_columns]
      .mean()
      .rename(columns=capability_labels)
      .round(2)
)
print(archetype_profile.to_string())


           研究/热点  AI创作  品牌/人设上下文  平台适配  多账号发布  审批/治理  质量/合规  分析/收件箱  Agent/工作流  本地/自托管
archetype                                                                             
AI营销内容平台     0.0   2.0       2.0   1.5    0.0    0.5    0.5     0.0        2.0     0.0
企业社媒平台       1.5   1.0       1.0   2.0    2.0    2.0    1.0     2.0        1.0     0.0
公众号创作工具      0.0   2.0       0.0   1.0    1.0    1.0    0.0     0.0        0.0     0.0
公众号运营插件      1.0   1.0       0.0   1.0    1.0    0.0    0.0     2.0        0.0     0.0
矩阵运营平台       0.5   1.0       0.0   1.0    2.0    1.0    0.5     2.0        0.5     1.0
社媒一体化平台      0.5   1.0       0.5   2.0    2.0    1.0    0.0     2.0        0.0     0.0
选题与内容情报      2.0   1.0       0.0   0.0    0.0    0.0    2.0     1.0        0.0     0.0
通用Agent平台    0.0   1.0       0.0   0.0    0.0    2.0    0.0     1.0        2.0     2.0
通用自动化平台      0.0   1.0       0.0   1.0    1.0    2.0    0.0     1.0        2.0     2.0


In [4]:
bundles = {
    '创作+发布': ['ai_creation', 'multi_account_publishing'],
    '发布+审批+分析': ['multi_account_publishing', 'approval_governance', 'analytics_inbox'],
    '品牌上下文+Agent工作流': ['brand_context', 'agent_workflows'],
    '研究+质量+发布': ['research_hotspots', 'quality_compliance', 'multi_account_publishing'],
    '审批+Agent+本地': ['approval_governance', 'agent_workflows', 'local_self_host'],
}

bundle_rows = []
for label, cols in bundles.items():
    matched = df.loc[(df[cols] == 2).all(axis=1), 'product'].tolist()
    bundle_rows.append({'bundle': label, 'core_match_count': len(matched), 'products': '、'.join(matched) or '无'})

bundle_df = pd.DataFrame(bundle_rows)
print(bundle_df.to_string(index=False))


        bundle  core_match_count                products
         创作+发布                 0                       无
      发布+审批+分析                 2 Hootsuite、Sprout Social
品牌上下文+Agent工作流                 2          Jasper、Copy.ai
      研究+质量+发布                 0                       无
   审批+Agent+本地                 2                Dify、n8n


## Takeaways

- AI 创作、平台适配、多账号发布、审批和分析都已有成熟供给，单点功能很难形成护城河。
- 品牌/人设上下文主要被 Jasper、Copy.ai 等营销平台做深；国内矩阵工具普遍较弱。
- 本地/自托管主要出现在 Dify、n8n 等通用平台，但它们不提供中文自媒体的领域闭环。
- “研究 → 有来源写作 → 质量/合规 → 人工审核 → 指定账号发布 → 结果回收”的完整组合在样本中没有直接同类。
- 最值得验证的不是更多平台数量，而是：可追溯内容工程、风险自适应审核、失败可恢复的浏览器技能，以及本地 Markdown/Agent 工作流。
